# B13 Agent - TPU v5e-8 Training Notebook

**Agent Name: b13**  
**Classification: PRIVATE**

Fine-tunes the B13 agent to call its 14 security tools using the smolagents format.

## Before running
1. Settings -> Accelerator = **TPU v5e-8** (v5e-8)
2. Settings -> Internet = **On**
3. Add-ons -> Secrets -> add **HF_TOKEN** (HuggingFace token)
4. '+ Add Data' -> attach **jhonmccarty/b13-agent-sft** and **jhonmccarty/b13-agent-code**
5. **Save Version -> Save & Run All (Commit)** (do NOT just Save)

In [ ]:
# Install dependencies
!pip install -q --upgrade 'smolagents>=1.26.0' 'transformers>=4.44.0' 'peft>=0.12.0' 'trl>=0.9.0' 'datasets>=2.18.0' 'accelerate>=0.33.0' 'huggingface-hub>=0.24.0' 'sentencepiece' 'protobuf'

In [ ]:
import os, sys, json, logging, glob

AGENT_NAME = "b13"
MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"
OUTPUT_DIR = f"/kaggle/working/{AGENT_NAME}-checkpoint"
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(AGENT_NAME)
print(f"=== {AGENT_NAME.upper()} AGENT TRAINING (TPU v5e-8) ===")
print(f"Model: {MODEL_ID}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Locate + copy the b13_agent code package from the attached code dataset
PROJECT_ROOT = "/kaggle/working/b13-agent"
os.makedirs(PROJECT_ROOT, exist_ok=True)
code_src = None
for cand in sorted(glob.glob('/kaggle/input/*/b13_agent')):
    if os.path.isdir(os.path.join(cand, 'tools')):
        code_src = os.path.dirname(cand)
        break
if code_src:
    print(f'Found b13_agent at: {code_src}')
    !cp -r {code_src}/b13_agent {PROJECT_ROOT}/
else:
    print('ERROR: b13_agent code dataset not found. Attach jhonmccarty/b13-agent-code')
sys.path.insert(0, PROJECT_ROOT)
os.environ['PYTHONPATH'] = PROJECT_ROOT
print('PYTHONPATH ->', PROJECT_ROOT)

In [ ]:
# Export the 14 tool schemas the agent must learn to call
from b13_agent.agent import export_tool_schemas, get_all_tools
schemas = export_tool_schemas()
print(f'B13 tools: {len(schemas)}')
for name, s in schemas.items():
    print(f'  {name}: {list(s["inputs"].keys())}')
json.dump(schemas, open(f'{RESULTS_DIR}/tool_schemas.json','w'), indent=2)
print('schemas saved')

In [ ]:
# Discover SFT parquet files (dedupe: skip HF data/ train-validation splits, keep the named SFT files)
KEEP = ['cve_funcs_tool_execution_sft','cve_tool_execution_sft','master_sft_enhanced_unified','master_sft_cleaned_unified']
all_parquet = sorted(glob.glob('/kaggle/input/**/*.parquet', recursive=True))
print(f'Found {len(all_parquet)} parquet files total')
train_files = [f for f in all_parquet if any(k in os.path.basename(f) for k in KEEP)]
# exclude the HF data/ split duplicates
train_files = [f for f in train_files if '/data/' not in f]
print(f'Selected {len(train_files)} SFT files:')
for f in train_files: print('  ', f)

In [ ]:
# Initialize TPU v5e-8 (torch_xla)
import torch
import torch_xla.core.xla_model as xm
device = xm.xla_device()
print('TPU device:', device)
print('TPU cores:', xm.xrt_world_size())

In [ ]:
# Load tokenizer + model onto TPU
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
HF_TOKEN = os.getenv('HF_TOKEN')
print('HF_TOKEN present:', bool(HF_TOKEN))
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True, token=HF_TOKEN
).to(device)
print(f'Loaded {MODEL_ID} | params {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.03,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
    modules_to_save=["embed_tokens","lm_head"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Build the training set with a chat-template formatting function (SFTTrainer needs text, not raw messages list)
from datasets import load_dataset, concatenate_datasets
import numpy as np

def to_list(m):
    return m.tolist() if isinstance(m, np.ndarray) else m

def formatting_func(examples):
    texts = []
    for msgs in examples['messages']:
        msgs = to_list(msgs)
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))
    return texts

datasets_list = []
for f in train_files:
    ds = load_dataset('parquet', data_files=f, split='train')
    datasets_list.append(ds)
    print(f'  loaded {os.path.basename(f)}: {len(ds):,} rows')
train_dataset = concatenate_datasets(datasets_list) if len(datasets_list) > 1 else datasets_list[0]
print(f'TOTAL training rows: {len(train_dataset):,}')
print('sample text preview:')
print(formatting_func(train_dataset[:1])[0][:300])

In [ ]:
# Training config (SFTTrainer with formatting_func)
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    max_seq_length=2048,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    optim="adamw_torch",
)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
print('Trainer ready. Starting training on TPU v5e-8...')

In [ ]:
# Train + save
trainer.train()
print('Training complete. Saving...')
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'=== B13 AGENT SAVED to {OUTPUT_DIR} ===')

In [ ]:
# Save training summary
artifacts = {
    "agent_name": AGENT_NAME,
    "model_id": MODEL_ID,
    "output_dir": OUTPUT_DIR,
    "tools": list(schemas.keys()),
    "total_tools": len(schemas),
    "training_rows": len(train_dataset),
    "tpu_cores": xm.xrt_world_size(),
    "epochs": sft_config.num_train_epochs,
}
json.dump(artifacts, open(f'{RESULTS_DIR}/training_summary.json','w'), indent=2)
print(json.dumps(artifacts, indent=2))